# ESG Media Sentiment Analysis (GDELT GKG)

**Raw dataset:** one 15-minute GDELT GKG batch (`20261117200000.gkg.csv`).

This notebook loads the raw GKG file, assigns the standard GDELT 2.0 GKG column headers, and displays the data.

## User Parameters

Edit the values in the cell below to control which data the pipeline processes. Then run all cells.

- **date_range**: inclusive `(start, end)` dates, ISO format (`"2026-11-17"`) or GDELT timestamp (`"20261117200000"`). Use `None` to leave a bound open.
- **continents**: e.g. `["North America", "Europe", "Asia"]`, or `["ALL"]` for no filter.
- **countries**: e.g. `["US", "GB", "ID"]` (ISO codes) or country names, or `["ALL"]` for no filter.
- **aggregation_freq**: `"1D"` (daily), `"1W"` (weekly), `"1M"` (monthly). Default `"1W"`.
- **min_articles**: minimum articles required for a (country, time) bucket to be colored on the map. Buckets below this are hidden as too noisy. Set `0` to disable.

In [1]:
# ============================================================
# USER PARAMETERS - edit these to control the pipeline
# ============================================================

# Inclusive date range (start, end). None = no bound.
# ISO format "2026-11-17" or GDELT timestamp "20261117200000".
date_range = ("20241117200000", "20251117200000")

# Target continents, e.g. ["North America", "Europe", "Asia"].
# Use ["ALL"] for no continent filter.
continents = ["Europe"]

# Target countries (ISO codes or names), e.g. ["US", "GB", "ID"].
# Use ["ALL"] for no country filter.
countries = ["ALL"]

# Aggregation frequency: "1D" daily, "1W" weekly, "1M" monthly.
aggregation_freq = "1M"

# Minimum articles required for a map bucket to be colored (0 = disable mask).
min_articles = 3

# Apply the user parameters to the config module
import config
config.date_range = date_range
config.continents = continents
config.countries = countries
config.aggregation_freq = aggregation_freq
config.MIN_ARTICLES = min_articles

print('date_range:', config.date_range)
print('continents:', config.continents)
print('countries:', config.countries)
print('aggregation_freq:', config.aggregation_freq)
print('min_articles:', config.MIN_ARTICLES)

date_range: ('20241117200000', '20251117200000')
continents: ['Europe']
countries: ['ALL']
aggregation_freq: 1M
min_articles: 3


In [2]:
# Download raw GDELT GKG batches based on the date_range above
# ============================================================
# DOWNLOAD SETTINGS - edit these to control downloading
# ============================================================
# Set to True to download data, False to skip (use existing files).
download_enabled = True

# Download mode:
#   'daily'  - one batch per day (23:45) in the date range
#   'all'    - every 15-minute batch in the date range (96/day)
#   'latest' - the most recent available batch (ignores date_range)
download_mode = 'daily'

# Set to True to re-download and overwrite files that already exist.
# False (default) keeps existing files and only downloads missing ones.
force_download = False

# Skip GDELT batches the server does not have (404) with a warning, instead
# of aborting. GDELT has occasional archive gaps (e.g. server outages).
# Set to False to stop on the first missing batch.
skip_missing = True

from download_raw import download_batch, download_date_range, download_all_batches, download_latest_batch

if download_enabled:
    if download_mode == 'daily':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=daily requires both date_range bounds (set them in the parameters cell)')
        paths = download_date_range(start, end, force=force_download, skip_missing=skip_missing)
    elif download_mode == 'all':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=all requires both date_range bounds (set them in the parameters cell)')
        paths = download_all_batches(start, end, force=force_download, skip_missing=skip_missing)
    elif download_mode == 'latest':
        paths = [download_latest_batch(force=force_download)]
    else:
        raise ValueError(f"Unknown download_mode: {download_mode!r} (use 'daily', 'all', or 'latest')")
    print('downloaded:', [p.name for p in paths])
else:
    print('download disabled - using existing raw files')

print('raw data dir:', config.raw_data_dir)
print('existing raw files:', [f.name for f in config.raw_data_dir.glob('*.gkg.csv')])


download disabled - using existing raw files
raw data dir: /home/theodoret/Downloads/Interdiscplinarly/main/data/raw
existing raw files: ['20250129234500.gkg.csv', '20241211234500.gkg.csv', '20250412234500.gkg.csv', '20250610234500.gkg.csv', '20251126234500.gkg.csv', '20250510234500.gkg.csv', '20250527234500.gkg.csv', '20251015234500.gkg.csv', '20250501234500.gkg.csv', '20250326234500.gkg.csv', '20250727234500.gkg.csv', '20250904234500.gkg.csv', '20251121234500.gkg.csv', '20250122234500.gkg.csv', '20250208234500.gkg.csv', '20250612234500.gkg.csv', '20250315234500.gkg.csv', '20250603234500.gkg.csv', '20241210234500.gkg.csv', '20251210234500.gkg.csv', '20251205234500.gkg.csv', '20250705234500.gkg.csv', '20250506234500.gkg.csv', '20250318234500.gkg.csv', '20250601234500.gkg.csv', '20250920234500.gkg.csv', '20251212234500.gkg.csv', '20250305234500.gkg.csv', '20250504234500.gkg.csv', '20250403234500.gkg.csv', '20250121234500.gkg.csv', '20250912234500.gkg.csv', '20251024234500.gkg.csv', '202

In [3]:
import sys
from pathlib import Path

# Make config.py importable
sys.path.insert(0, str(Path.cwd()))

import config
import pandas as pd

print('raw dir:', config.raw_data_dir)

raw dir: /home/theodoret/Downloads/Interdiscplinarly/main/data/raw


In [4]:
# Locate and load the raw GKG files via the fetch_raw module
from fetch_raw import fetch_raw_gkg, find_raw_gkg_files

df, raw_paths, missing_days = fetch_raw_gkg(date_range=config.date_range)
print('raw GKG files found:', [f.name for f in find_raw_gkg_files()])
print('loaded files:', len(raw_paths), '->', [p.name for p in raw_paths])
print('missing days (no raw file):', missing_days)
print('shape:', df.shape)
print('columns:', list(df.columns))


raw GKG files found: ['20241117234500.gkg.csv', '20241118234500.gkg.csv', '20241119234500.gkg.csv', '20241120234500.gkg.csv', '20241121234500.gkg.csv', '20241122234500.gkg.csv', '20241123234500.gkg.csv', '20241124234500.gkg.csv', '20241125234500.gkg.csv', '20241126234500.gkg.csv', '20241127234500.gkg.csv', '20241128234500.gkg.csv', '20241129234500.gkg.csv', '20241130234500.gkg.csv', '20241201234500.gkg.csv', '20241202234500.gkg.csv', '20241203234500.gkg.csv', '20241204234500.gkg.csv', '20241205234500.gkg.csv', '20241206234500.gkg.csv', '20241207234500.gkg.csv', '20241208234500.gkg.csv', '20241209234500.gkg.csv', '20241210234500.gkg.csv', '20241211234500.gkg.csv', '20241212234500.gkg.csv', '20241213234500.gkg.csv', '20241214234500.gkg.csv', '20241215234500.gkg.csv', '20241216234500.gkg.csv', '20241217234500.gkg.csv', '20241218234500.gkg.csv', '20241219234500.gkg.csv', '20241220234500.gkg.csv', '20241221234500.gkg.csv', '20241222234500.gkg.csv', '20241223234500.gkg.csv', '20241224234500.

In [5]:
# Preprocess: NA handling, dedup, and persistence via the preprocess module
from preprocess import preprocess, save_processed

df_clean = preprocess(df)
print('clean shape:', df_clean.shape)

clean_path = save_processed(df_clean)
print('saved to:', clean_path)

preprocess: dropped 0 rows with missing DATE
preprocess: 298397 rows without V2Tone (kept for volume stats)
preprocess: 0 rows flagged missing_identifier
preprocess: dropped 0 duplicate rows on ['DocumentIdentifier', 'DATE']
clean shape: (465406, 28)
preprocess: saved 465406 rows x 8 cols -> /home/theodoret/Downloads/Interdiscplinarly/main/data/processed/gkg_clean.csv
saved to: /home/theodoret/Downloads/Interdiscplinarly/main/data/processed/gkg_clean.csv


In [6]:
# ESG classification: multi-label tagging via the esg_classify module
from esg_classify import classify

df_esg = classify(df_clean)
print('esg shape:', df_esg.shape)

pillar_cols = [f'is_{p}' for p in config.esg_categories]
print('tagged counts:')
print(df_esg[pillar_cols].sum())
print('multi-label rows:', (df_esg[pillar_cols].sum(axis=1) > 1).sum())
print('unclassified rows:', (df_esg[pillar_cols].sum(axis=1) == 0).sum())

esg shape: (465406, 32)
tagged counts:
is_Environmental    175536
is_Social           326371
is_Governance       395838
dtype: int64
multi-label rows: 334707
unclassified rows: 57452


In [7]:
# Tone / sentiment extraction via the tone module
from tone import extract_tone

df_tone = extract_tone(df_esg)
print('tone shape:', df_tone.shape)

tone_cols = ['tone', 'positive', 'negative', 'polarity']
print('rows with tone:', df_tone['tone'].notna().sum())
print(df_tone[tone_cols].describe())

tone shape: (465406, 36)
rows with tone: 167009
                tone       positive       negative       polarity
count  167009.000000  167009.000000  167009.000000  167009.000000
mean        2.650318       3.467394       8.226611      -4.759217
std         1.464824       3.846851      10.090946       8.589165
min         1.000000       0.000000       0.000000     -30.000000
25%         1.000000       0.000000       0.000000     -10.000000
50%         4.000000       3.000000       2.000000       0.000000
75%         4.000000       6.000000      16.000000       0.000000
max         4.000000      12.000000      31.000000      12.000000


In [8]:
# Media source geographic attribution via the geo_attr module
from geo_attr import attribute, filter_by_countries

df_geo = attribute(df_tone)
print('geo shape (before filter):', df_geo.shape)

# Apply user country / continent filters
df_geo = filter_by_countries(df_geo, countries=config.countries, continents=config.continents)
print('geo shape (after filter):', df_geo.shape)

print('source_country coverage:', df_geo['source_country'].notna().sum(), '/', len(df_geo))
print('location_country coverage:', df_geo['location_country'].notna().sum(), '/', len(df_geo))
print('attributed_country coverage:', (df_geo['attributed_country'] != config.UNKNOWN_SOURCE).sum(), '/', len(df_geo))
print('\ntop attributed countries:')
print(df_geo['attributed_country'].value_counts().head(10))

geo shape (before filter): (465406, 40)
geo shape (after filter): (61338, 40)
source_country coverage: 34844 / 61338
location_country coverage: 54062 / 61338
attributed_country coverage: 61338 / 61338

top attributed countries:
attributed_country
United Kingdom    38698
Ireland            3630
France             2842
Russia             2525
Germany            2335
Italy              2108
Ukraine            1998
Spain              1201
Switzerland         662
Greece              626
Name: count, dtype: int64


In [9]:
# Metric aggregation: ESG x country x time window via the aggregate module
from aggregate import aggregate, save_aggregated

df_agg = aggregate(df_geo, freq=config.aggregation_freq)
print('agg shape:', df_agg.shape)
print('freq:', config.aggregation_freq)
print('categories:', sorted(df_agg['esg_category'].unique()))
print('countries:', df_agg['country'].nunique())
print('date range:', df_agg['date'].min(), '->', df_agg['date'].max())
print('\nhead:')
print(df_agg.head(10).to_string(index=False))

agg_path = save_aggregated(df_agg)
print('saved to:', agg_path)

agg shape: (2017, 8)
freq: 1M
categories: ['Composite', 'Environmental', 'Governance', 'Social']
countries: 46
date range: 2024-11-01 00:00:00 -> 2025-11-01 00:00:00

head:
      date country  esg_category  tone_mean  tone_median  tone_std  n_with_tone  n_articles
2024-11-01 Albania     Composite        NaN          NaN       NaN            0           1
2024-11-01 Albania    Governance        NaN          NaN       NaN            0           1
2024-11-01 Albania        Social        NaN          NaN       NaN            0           1
2024-11-01 Austria     Composite       1.50          1.0  1.224745            6          12
2024-11-01 Austria Environmental       1.75          1.0  1.500000            4           4
2024-11-01 Austria    Governance       1.50          1.0  1.224745            6          12
2024-11-01 Austria        Social       1.50          1.0  1.224745            6          11
2024-11-01 Belarus     Composite       4.00          4.0       NaN            1           2

In [10]:
# Interactive world map visualization via the viz_map module
from viz_map import build_choropleth, build_date_slider_figure, build_category_selector_figure, build_time_series_figure, save_figure

# Latest-date choropleth for each ESG category
for cat in config.esg_categories + ['Composite']:
    fig = build_choropleth(df_agg, category=cat)
    save_figure(fig, f'map_{cat.lower()}')

# Date-slider figure (Composite)
slider_fig = build_date_slider_figure(df_agg, category='Composite')
save_figure(slider_fig, 'map_composite_slider')

# Category-switcher figure (dropdown: E / S / G / Composite)
selector_fig = build_category_selector_figure(df_agg)
save_figure(selector_fig, 'map_category_selector')

# Time-series figure: category dropdown + date slider (change over time)
time_fig = build_time_series_figure(df_agg)
save_figure(time_fig, 'map_time_series')

# Show the time-series map inline
build_time_series_figure(df_agg)

viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_environmental.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_social.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_governance.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_composite.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_composite_slider.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_category_selector.html
viz: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/output/map_time_series.html


In [11]:
# Show the data
df_esg.head()

,GKGRecordID,DATE,NumArticles,SourceCommonName,DocumentIdentifier,V1Counts,V2Counts,V1Themes,V2Themes,V1Locations,...,V2SocialImage,V1Quotations,V2Quotations,V1AllNames,V2AllNames,missing_identifier,is_Environmental,is_Social,is_Governance,esg_pillars
0,20241117234500-0,20241117234500,1,mycowichanvalleynow.com,https://www.mycowichanvalleynow.com/86873/news...,,,GENERAL_HEALTH;MEDICAL;TAX_FNCACT;TAX_FNCACT_N...,"CRISISLEX_C03_WELLBEING_HEALTH,102;CRISISLEX_C...","4#Nanaimo, British Columbia, Canada#CA#CA02#49...",...,,"Jordan Gerrard,193;Inquiry Committee,362",,,<PAGE_AUTHORS>Grant Warkentin</PAGE_AUTHORS><P...,False,False,True,True,Governance;Social
1,20241117234500-1,20241117234500,1,bunburymail.com.au,https://www.bunburymail.com.au/story/8822244/c...,,,CRISISLEX_CRISISLEXREC;MANMADE_DISASTER;MANMAD...,"TAX_FNCACT_SECRETARY,1707;TAX_FNCACT_METEOROLO...","4#Dubbo, New South Wales, Australia#AS#AS02#-3...",...,,"Shire Mayor Jasen Ramien,920;Central West,1326...","278,calls for emergency help,19;278,calls for ...",,<PAGE_PRECISEPUBTIMESTAMP>20241117220100</PAGE...,False,True,False,True,Environmental;Governance
2,20241117234500-2,20241117234500,1,inverelltimes.com.au,https://www.inverelltimes.com.au/story/8822287...,,,CRISISLEX_CRISISLEXREC;,"CRISISLEX_CRISISLEXREC,346;",,...,,"Jack Quick,122;CarExpert Podcast,597;Apple Pod...",,,<PAGE_LINKS>https://www.carexpert.com.au;https...,False,False,False,False,
3,20241117234500-3,20241117234500,1,dailymail.co.uk,https://www.dailymail.co.uk/tvshowbiz/article-...,,,TAX_FNCACT;TAX_FNCACT_JUDGE;TAX_FNCACT_DANCER;,"TAX_FNCACT_DANCER,4722;TAX_FNCACT_DANCER,4793;...",1#United Kingdom#UK#UK#54#-4#UK,...,,"Coleen Rooney,174;Danny Jones,191;Barry McGuig...","2,Strictly glitterballs under her,3875;6000000...",,<PAGE_LINKS>https://www.dailymail.co.uk/tvshow...,False,False,False,True,Governance
4,20241117234500-4,20241117234500,1,go.com,https://abcnews.go.com/Politics/biden-makes-hi...,,,LEADER;UNGP_FORESTS_RIVERS_OCEANS;,"LEADER,9;LEADER,206;LEADER,1337;LEADER,2200;UN...",1#Japan#JA#JA#36#138#JA;1#South Korea#KS#KS#37...,...,1896|225||Not when countries around the world ...,"President Joe Biden,20;Donald Trump,661;South ...","10,soccer fields worth of,424;",,<PAGE_LINKS>https://abcnews.go.com/Politics/ai...,False,False,True,False,Social


In [12]:
# Focus on the core columns from the plan
core = ['DATE', 'DocumentIdentifier', 'V2Themes', 'V2Locations', 'V2Tone']
df[core].head(10)

,DATE,DocumentIdentifier,V2Themes,V2Locations,V2Tone
0,20241117234500,https://www.mycowichanvalleynow.com/86873/news...,"CRISISLEX_C03_WELLBEING_HEALTH,102;CRISISLEX_C...","4#Nanaimo, British Columbia, Canada#CA#CA02#12...",
1,20241117234500,https://www.bunburymail.com.au/story/8822244/c...,"TAX_FNCACT_SECRETARY,1707;TAX_FNCACT_METEOROLO...","4#Mudgee, New South Wales, Australia#AS#AS02#5...",
2,20241117234500,https://www.inverelltimes.com.au/story/8822287...,"CRISISLEX_CRISISLEXREC,346;",,
3,20241117234500,https://www.dailymail.co.uk/tvshowbiz/article-...,"TAX_FNCACT_DANCER,4722;TAX_FNCACT_DANCER,4793;...",1#United Kingdom#UK#UK##54#-4#UK#5166;1#Britis...,1#0#0#1986#5540;1#0#0#2002#6502;1#0#0#2013#655...
4,20241117234500,https://abcnews.go.com/Politics/biden-makes-hi...,"LEADER,9;LEADER,206;LEADER,1337;LEADER,2200;UN...",1#Chinese#CH#CH##35#105#CH#1327;1#American#US#...,
5,20241117234500,https://cheknews.ca/duncan-resident-fined-6k-f...,"SOC_AIDASSISTANCE,102;SOC_AIDASSISTANCE,853;TA...",,
6,20241117234500,https://www.si.com/nfl/mics-caught-derek-carr-...,,"3#Denver, Colorado, United States#US#USCO#CO03...",1#0#0#2014#468
7,20241117234500,https://www.thetelegraphandargus.co.uk/news/na...,"LEADER,43;LEADER,292;LEADER,1292;LEADER,1611;L...",1#British#UK#UK##54#-4#UK#28;1#China#CH#CH##35...,
8,20241117234500,https://www.smartcompany.com.au/finance/tax/ta...,"EPU_UNCERTAINTY,1058;TAX_FNCACT_SPECIALIST,257...",1#Australia#AS#AS##-25#135#AS#73,
9,20241117234500,https://www.theyeshivaworld.com/news/israel-ne...,"PROTEST,476;SECURITY_SERVICES,63;SECURITY_SERV...",,
